In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline


In [ ]:
# ============================================================
# 1. Paths
# ============================================================

INPUT_PATH = Path("../data/processed/neutral_scoring_prepared/neutral_scoring_input_v1.tsv")

OUTPUT_DIR = Path("../results/neutral_scoring_results")
OUTPUT_DIR.mkdir(exist_ok=True)

SCORED_OUTPUT = OUTPUT_DIR / "neutral_deviation_scores_v1.tsv"
THRESHOLD_OUTPUT = OUTPUT_DIR / "neutral_deviation_thresholds_v1.tsv"
ENRICHMENT_OUTPUT = OUTPUT_DIR / "posthoc_group_enrichment_v1.tsv"

# ============================================================
# 2. Load prepared input
# ============================================================

df = pd.read_csv(INPUT_PATH, sep="\t", low_memory=False)

print("Input shape:", df.shape)

print("\nAnalysis group distribution:")
print(df["analysis_group"].value_counts(dropna=False))

# ============================================================
# 3. Define feature sets
# ============================================================

feature_sets = {
    "mlc_only": [
        "mlc_score",
    ],

    "population_only": [
        "gnomad_observed",
        "helix_observed",
        "observed_in_any_population_db",
        "observed_in_both_population_dbs",

        "gnomad_homoplasmic_af",
        "gnomad_heteroplasmic_af",
        "gnomad_combined_af_simple",
        "helix_af_hom",
        "helix_af_het",

        "pop_af_hom_max",
        "pop_af_het_max",
        "pop_af_max",
        "pop_af_hom_sum",
        "pop_af_het_sum",
        "pop_af_sum",

        "log10_gnomad_homoplasmic_af",
        "log10_gnomad_heteroplasmic_af",
        "log10_gnomad_combined_af_simple",
        "log10_helix_af_hom",
        "log10_helix_af_het",
        "log10_pop_af_hom_max",
        "log10_pop_af_het_max",
        "log10_pop_af_max",
        "log10_pop_af_hom_sum",
        "log10_pop_af_het_sum",
        "log10_pop_af_sum",

        "gnomad_max_heteroplasmy",
        "helix_mean_arf",
        "helix_max_arf",
        "pop_max_heteroplasmy_or_arf",
    ],
}

feature_sets["combined_mlc_population"] = (
    feature_sets["mlc_only"] + feature_sets["population_only"]
)

# Remove duplicates
for name in feature_sets:
    feature_sets[name] = list(dict.fromkeys(feature_sets[name]))

SyntaxError: invalid syntax (4004646466.py, line 1)

In [36]:
# ============================================================
# 3. Define non-feature columns
# ============================================================

id_cols = [
    "variant_id",
    "position",
    "reference",
    "alternate",
]

target_cols = [
    "target",
    "validation_label",
]

non_feature_cols = id_cols + target_cols

In [37]:
# ============================================================
# 4. Define feature sets
# ============================================================

# Model A: only mlc_score
mlc_only_features = [
    "mlc_score",
]

# Model B: all features from core table
core_features = [
    col for col in core_df.columns
    if col not in non_feature_cols
]

# Model C: all features from extended table
extended_features = [
    col for col in extended_df.columns
    if col not in non_feature_cols
]

print("Number of MLC-only features:", len(mlc_only_features))
print("Number of core features:", len(core_features))
print("Number of extended features:", len(extended_features))

print("\nCore features:")
for col in core_features:
    print("-", col)


Number of MLC-only features: 1
Number of core features: 46
Number of extended features: 167

Core features:
- mlc_score
- gnomad_observed
- helix_observed
- observed_in_any_population_db
- observed_in_both_population_dbs
- AN
- gnomad_homoplasmic_ac
- gnomad_heteroplasmic_ac
- helix_counts_hom
- helix_counts_het
- gnomad_homoplasmic_af
- gnomad_heteroplasmic_af
- gnomad_combined_af_simple
- helix_af_hom
- helix_af_het
- gnomad_max_heteroplasmy
- helix_mean_arf
- helix_max_arf
- pop_max_heteroplasmy_or_arf
- pop_hom_ac_sum
- pop_het_ac_sum
- pop_ac_sum
- pop_af_hom_max
- pop_af_het_max
- pop_af_max
- pop_af_hom_sum
- pop_af_het_sum
- pop_af_sum
- log10_gnomad_homoplasmic_ac
- log10_gnomad_heteroplasmic_ac
- log10_helix_counts_hom
- log10_helix_counts_het
- log10_pop_hom_ac_sum
- log10_pop_het_ac_sum
- log10_pop_ac_sum
- log10_gnomad_homoplasmic_af
- log10_gnomad_heteroplasmic_af
- log10_gnomad_combined_af_simple
- log10_helix_af_hom
- log10_helix_af_het
- log10_pop_af_hom_max
- log10_po

In [38]:
print("\nFirst 20 extended features:")
for col in extended_features[:20]:
    print("-", col)


First 20 extended features:
- mlc_score
- gnomad_observed
- helix_observed
- observed_in_any_population_db
- observed_in_both_population_dbs
- AN
- gnomad_homoplasmic_ac
- gnomad_heteroplasmic_ac
- helix_counts_hom
- helix_counts_het
- gnomad_homoplasmic_af
- gnomad_heteroplasmic_af
- gnomad_combined_af_simple
- helix_af_hom
- helix_af_het
- gnomad_max_heteroplasmy
- helix_mean_arf
- helix_max_arf
- pop_max_heteroplasmy_or_arf
- pop_hom_ac_sum


In [39]:
# ============================================================
# 5. Sanity checks
# ============================================================

def check_feature_table(df, feature_cols, name):
    print(f"\nChecking {name}")

    missing = df[feature_cols].isna().sum()
    missing = missing[missing > 0]

    if len(missing) > 0:
        print("Columns with missing values:")
        print(missing)
        raise ValueError(f"{name} contains missing values in features.")
    else:
        print("No missing values.")

    non_numeric = [
        col for col in feature_cols
        if not pd.api.types.is_numeric_dtype(df[col])
    ]

    if len(non_numeric) > 0:
        print("Non-numeric columns:")
        print(non_numeric)
        raise ValueError(f"{name} contains non-numeric feature columns.")
    else:
        print("All features are numeric.")

    if df["target"].isna().any():
        raise ValueError(f"{name} contains missing target values.")

    print("Target distribution:")
    print(df["target"].value_counts())


check_feature_table(core_df, mlc_only_features, "MLC-only table")
check_feature_table(core_df, core_features, "Core table")
check_feature_table(extended_df, extended_features, "Extended table")


Checking MLC-only table
No missing values.
All features are numeric.
Target distribution:
target
0    8183
1      88
Name: count, dtype: int64

Checking Core table
No missing values.
All features are numeric.
Target distribution:
target
0    8183
1      88
Name: count, dtype: int64

Checking Extended table
No missing values.
All features are numeric.
Target distribution:
target
0    8183
1      88
Name: count, dtype: int64


In [40]:
# ============================================================
# 6. Shared train/test split
# ============================================================

RANDOM_STATE = 42
TEST_SIZE = 0.25

y = core_df["target"].astype(int)

train_idx, test_idx = train_test_split(
    core_df.index,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Train size:", len(train_idx))
print("Test size:", len(test_idx))

print("\nTrain target distribution:")
print(core_df.loc[train_idx, "target"].value_counts())

print("\nTest target distribution:")
print(core_df.loc[test_idx, "target"].value_counts())

Train size: 6203
Test size: 2068

Train target distribution:
target
0    6137
1      66
Name: count, dtype: int64

Test target distribution:
target
0    2046
1      22
Name: count, dtype: int64


In [41]:
# ============================================================
# 7. Evaluation helper
# ============================================================

def evaluate_binary_classifier(model, X_train, y_train, X_test, y_test, model_name):
    model.fit(X_train, y_train)

    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    else:
        y_score = model.decision_function(X_test)

    y_pred_default = (y_score >= 0.5).astype(int)

    roc_auc = roc_auc_score(y_test, y_score)
    pr_auc = average_precision_score(y_test, y_score)
    accuracy = accuracy_score(y_test, y_pred_default)
    balanced_acc = balanced_accuracy_score(y_test, y_pred_default)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_default).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan

    result = {
        "model": model_name,
        "roc_auc": roc_auc,
        "pr_auc": pr_auc,
        "accuracy_threshold_0_5": accuracy,
        "balanced_accuracy_threshold_0_5": balanced_acc,
        "sensitivity_threshold_0_5": sensitivity,
        "specificity_threshold_0_5": specificity,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "tp": tp,
    }

    print("\n" + "=" * 80)
    print(model_name)
    print("=" * 80)
    print("ROC-AUC:", round(roc_auc, 4))
    print("PR-AUC:", round(pr_auc, 4))
    print("Accuracy:", round(accuracy, 4))
    print("Balanced accuracy:", round(balanced_acc, 4))
    print("Sensitivity:", round(sensitivity, 4))
    print("Specificity:", round(specificity, 4))
    print("\nConfusion matrix at threshold 0.5:")
    print(confusion_matrix(y_test, y_pred_default))
    print("\nClassification report:")
    print(classification_report(y_test, y_pred_default, digits=4))

    predictions = pd.DataFrame({
        "y_true": y_test.values,
        "y_score": y_score,
        "y_pred_threshold_0_5": y_pred_default,
    }, index=y_test.index)

    return model, result, predictions

In [42]:
# ============================================================
# 8. Logistic regression models
# ============================================================

models_results = []
predictions_dict = {}

# ----------------------------
# Model A: mlc_score only
# ----------------------------

X_train = core_df.loc[train_idx, mlc_only_features]
X_test = core_df.loc[test_idx, mlc_only_features]
y_train = core_df.loc[train_idx, "target"].astype(int)
y_test = core_df.loc[test_idx, "target"].astype(int)

logreg_mlc = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

fitted_logreg_mlc, result, pred = evaluate_binary_classifier(
    logreg_mlc,
    X_train,
    y_train,
    X_test,
    y_test,
    "logistic_regression_mlc_score_only",
)

models_results.append(result)
predictions_dict["logistic_regression_mlc_score_only"] = pred


# ----------------------------
# Model B: core features
# ----------------------------

X_train = core_df.loc[train_idx, core_features]
X_test = core_df.loc[test_idx, core_features]

logreg_core = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=5000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

fitted_logreg_core, result, pred = evaluate_binary_classifier(
    logreg_core,
    X_train,
    y_train,
    X_test,
    y_test,
    "logistic_regression_core_v2",
)

models_results.append(result)
predictions_dict["logistic_regression_core_v2"] = pred


# ----------------------------
# Model C: extended features
# ----------------------------

X_train = extended_df.loc[train_idx, extended_features]
X_test = extended_df.loc[test_idx, extended_features]

logreg_extended = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=10000,
        class_weight="balanced",
        random_state=RANDOM_STATE,
    )),
])

fitted_logreg_extended, result, pred = evaluate_binary_classifier(
    logreg_extended,
    X_train,
    y_train,
    X_test,
    y_test,
    "logistic_regression_extended_v2",
)

models_results.append(result)
predictions_dict["logistic_regression_extended_v2"] = pred


logistic_regression_mlc_score_only
ROC-AUC: 0.8848
PR-AUC: 0.0491
Accuracy: 0.8501
Balanced accuracy: 0.8343
Sensitivity: 0.8182
Specificity: 0.8504

Confusion matrix at threshold 0.5:
[[1740  306]
 [   4   18]]

Classification report:
              precision    recall  f1-score   support

           0     0.9977    0.8504    0.9182      2046
           1     0.0556    0.8182    0.1040        22

    accuracy                         0.8501      2068
   macro avg     0.5266    0.8343    0.5111      2068
weighted avg     0.9877    0.8501    0.9095      2068


logistic_regression_core_v2
ROC-AUC: 0.9709
PR-AUC: 0.3781
Accuracy: 0.9221
Balanced accuracy: 0.9382
Sensitivity: 0.9545
Specificity: 0.9218

Confusion matrix at threshold 0.5:
[[1886  160]
 [   1   21]]

Classification report:
              precision    recall  f1-score   support

           0     0.9995    0.9218    0.9591      2046
           1     0.1160    0.9545    0.2069        22

    accuracy                         0.922

In [43]:
# ============================================================
# 9. Random Forest models
# ============================================================

# ----------------------------
# Random Forest: core
# ----------------------------

rf_core = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

X_train = core_df.loc[train_idx, core_features]
X_test = core_df.loc[test_idx, core_features]

fitted_rf_core, result, pred = evaluate_binary_classifier(
    rf_core,
    X_train,
    y_train,
    X_test,
    y_test,
    "random_forest_core_v2",
)

models_results.append(result)
predictions_dict["random_forest_core_v2"] = pred


# ----------------------------
# Random Forest: extended
# ----------------------------

rf_extended = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced",
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

X_train = extended_df.loc[train_idx, extended_features]
X_test = extended_df.loc[test_idx, extended_features]

fitted_rf_extended, result, pred = evaluate_binary_classifier(
    rf_extended,
    X_train,
    y_train,
    X_test,
    y_test,
    "random_forest_extended_v2",
)

models_results.append(result)
predictions_dict["random_forest_extended_v2"] = pred


random_forest_core_v2
ROC-AUC: 0.9572
PR-AUC: 0.505
Accuracy: 0.9908
Balanced accuracy: 0.748
Sensitivity: 0.5
Specificity: 0.9961

Confusion matrix at threshold 0.5:
[[2038    8]
 [  11   11]]

Classification report:
              precision    recall  f1-score   support

           0     0.9946    0.9961    0.9954      2046
           1     0.5789    0.5000    0.5366        22

    accuracy                         0.9908      2068
   macro avg     0.7868    0.7480    0.7660      2068
weighted avg     0.9902    0.9908    0.9905      2068


random_forest_extended_v2
ROC-AUC: 0.9844
PR-AUC: 0.5375
Accuracy: 0.9884
Balanced accuracy: 0.7918
Sensitivity: 0.5909
Specificity: 0.9927

Confusion matrix at threshold 0.5:
[[2031   15]
 [   9   13]]

Classification report:
              precision    recall  f1-score   support

           0     0.9956    0.9927    0.9941      2046
           1     0.4643    0.5909    0.5200        22

    accuracy                         0.9884      2068
   macro

In [44]:
RESULTS_DIR

PosixPath('../results/classification_results')

In [45]:
# ============================================================
# 10. Save model comparison
# ============================================================

results_df = pd.DataFrame(models_results)

results_df = results_df.sort_values(
    by=["roc_auc", "pr_auc"],
    ascending=False,
)

results_path = RESULTS_DIR / "model_comparison_v1.tsv"
results_df.to_csv(results_path, sep="\t", index=False)

print("\nModel comparison:")
print(results_df)

print("\nSaved:")
print(results_path)


Model comparison:
                                model   roc_auc    pr_auc  \
4           random_forest_extended_v2  0.984449  0.537515   
2     logistic_regression_extended_v2  0.971230  0.516310   
1         logistic_regression_core_v2  0.970897  0.378111   
3               random_forest_core_v2  0.957156  0.504959   
0  logistic_regression_mlc_score_only  0.884831  0.049113   

   accuracy_threshold_0_5  balanced_accuracy_threshold_0_5  \
4                0.988395                         0.791789   
2                0.957447                         0.866080   
1                0.922147                         0.938172   
3                0.990812                         0.748045   
0                0.850097                         0.834311   

   sensitivity_threshold_0_5  specificity_threshold_0_5    tn   fp  fn  tp  
4                   0.590909                   0.992669  2031   15   9  13  
2                   0.772727                   0.959433  1963   83   5  17  
1         

In [46]:
# ============================================================
# 11. Save test predictions
# ============================================================

test_metadata = core_df.loc[
    test_idx,
    id_cols + ["target", "validation_label"]
].copy()

for model_name, pred_df in predictions_dict.items():
    out = test_metadata.join(pred_df)

    out_path = RESULTS_DIR / f"test_predictions_{model_name}.tsv"
    out.to_csv(out_path, sep="\t", index=False)

    print("Saved:", out_path)

Saved: ../results/classification_results/test_predictions_logistic_regression_mlc_score_only.tsv
Saved: ../results/classification_results/test_predictions_logistic_regression_core_v2.tsv
Saved: ../results/classification_results/test_predictions_logistic_regression_extended_v2.tsv
Saved: ../results/classification_results/test_predictions_random_forest_core_v2.tsv
Saved: ../results/classification_results/test_predictions_random_forest_extended_v2.tsv


In [47]:
# ============================================================
# 12. Cross-validation helper
# ============================================================

def run_cross_validation(model, df, feature_cols, model_name, n_splits=5):
    X = df[feature_cols]
    y = df["target"].astype(int)

    cv = StratifiedKFold(
        n_splits=n_splits,
        shuffle=True,
        random_state=RANDOM_STATE,
    )

    scoring = {
        "roc_auc": "roc_auc",
        "average_precision": "average_precision",
        "balanced_accuracy": "balanced_accuracy",
        "accuracy": "accuracy",
    }

    cv_result = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
    )

    row = {
        "model": model_name,
    }

    for metric in scoring.keys():
        values = cv_result[f"test_{metric}"]
        row[f"{metric}_mean"] = np.mean(values)
        row[f"{metric}_std"] = np.std(values)

    return row

In [48]:
# ============================================================
# 13. Run cross-validation
# ============================================================

cv_rows = []

cv_rows.append(
    run_cross_validation(
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=1000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),
        core_df,
        mlc_only_features,
        "cv_logistic_regression_mlc_score_only",
    )
)

cv_rows.append(
    run_cross_validation(
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=5000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),
        core_df,
        core_features,
        "cv_logistic_regression_core_v2",
    )
)

cv_rows.append(
    run_cross_validation(
        Pipeline([
            ("scaler", StandardScaler()),
            ("model", LogisticRegression(
                max_iter=10000,
                class_weight="balanced",
                random_state=RANDOM_STATE,
            )),
        ]),
        extended_df,
        extended_features,
        "cv_logistic_regression_extended_v2",
    )
)

cv_rows.append(
    run_cross_validation(
        RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        core_df,
        core_features,
        "cv_random_forest_core_v2",
    )
)

cv_rows.append(
    run_cross_validation(
        RandomForestClassifier(
            n_estimators=500,
            min_samples_leaf=2,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        extended_df,
        extended_features,
        "cv_random_forest_extended_v2",
    )
)

cv_results_df = pd.DataFrame(cv_rows)

cv_results_df = cv_results_df.sort_values(
    by=["roc_auc_mean", "average_precision_mean"],
    ascending=False,
)

cv_results_path = RESULTS_DIR / "cross_validation_results_v1.tsv"
cv_results_df.to_csv(cv_results_path, sep="\t", index=False)

print("\nCross-validation results:")
print(cv_results_df)

print("\nSaved:")
print(cv_results_path)


Cross-validation results:
                                   model  roc_auc_mean  roc_auc_std  \
2     cv_logistic_regression_extended_v2      0.975634     0.010475   
4           cv_random_forest_extended_v2      0.967105     0.011519   
1         cv_logistic_regression_core_v2      0.959986     0.013052   
3               cv_random_forest_core_v2      0.956369     0.012236   
0  cv_logistic_regression_mlc_score_only      0.901799     0.014720   

   average_precision_mean  average_precision_std  balanced_accuracy_mean  \
2                0.475997               0.047915                0.884302   
4                0.624002               0.118180                0.786915   
1                0.440041               0.102423                0.884406   
3                0.466371               0.112007                0.769738   
0                0.073677               0.018028                0.826762   

   balanced_accuracy_std  accuracy_mean  accuracy_std  
2               0.061089       0.

In [49]:
# ============================================================
# 14. Random Forest feature importance
# ============================================================

rf_extended_importance = pd.DataFrame({
    "feature": extended_features,
    "importance": fitted_rf_extended.feature_importances_,
})

rf_extended_importance = rf_extended_importance.sort_values(
    by="importance",
    ascending=False,
)

importance_path = RESULTS_DIR / "feature_importance_random_forest_extended_v2.tsv"
rf_extended_importance.to_csv(importance_path, sep="\t", index=False)

print("\nTop 30 RF extended features:")
print(rf_extended_importance.head(30))

print("\nSaved:")
print(importance_path)


Top 30 RF extended features:
                                            feature  importance
0                                         mlc_score    0.127789
107                  consequence_synonymous_variant    0.098413
53                         gene_constraint_expected    0.038559
32                             log10_pop_hom_ac_sum    0.030033
52                         gene_constraint_observed    0.029165
19                                   pop_hom_ac_sum    0.026301
30                           log10_helix_counts_hom    0.025014
22                                   pop_af_hom_max    0.024634
25                                   pop_af_hom_sum    0.023632
97                     consequence_missense_variant    0.022575
55                         gene_constraint_lower_ci    0.021839
43                             log10_pop_af_hom_sum    0.021540
13                                     helix_af_hom    0.021158
8                                  helix_counts_hom    0.021101
40        

In [50]:
# ============================================================
# 15. Logistic regression coefficients for extended model
# ============================================================

logreg_model = fitted_logreg_extended.named_steps["model"]

coef_df = pd.DataFrame({
    "feature": extended_features,
    "coefficient": logreg_model.coef_[0],
})

coef_df["abs_coefficient"] = coef_df["coefficient"].abs()

coef_df = coef_df.sort_values(
    by="abs_coefficient",
    ascending=False,
)

coef_path = RESULTS_DIR / "coefficients_logistic_regression_extended_v2.tsv"
coef_df.to_csv(coef_path, sep="\t", index=False)

print("\nTop 30 logistic regression coefficients:")
print(coef_df.head(30))

print("\nSaved:")
print(coef_path)


Top 30 logistic regression coefficients:
                                            feature  coefficient  \
53                         gene_constraint_expected    -2.574081   
42                                 log10_pop_af_max    -2.221595   
15                          gnomad_max_heteroplasmy     1.801621   
55                         gene_constraint_lower_ci     1.748195   
23                                   pop_af_het_max    -1.650689   
45                                 log10_pop_af_sum    -1.544019   
107                  consequence_synonymous_variant    -1.510734   
100  consequence_non_coding_transcript_exon_variant     1.383502   
164         gene_constraint_consequence_RNA_variant     1.382018   
16                                   helix_mean_arf    -1.325132   
40                             log10_pop_af_hom_max    -1.317873   
54                          gene_constraint_obs_exp     1.305754   
41                             log10_pop_af_het_max     1.277969   
136   

In [51]:

RESULTS_DIR = Path("../results/classification_results")
PREPARED_DIR = Path("../results/classification_prepared")
VALIDATION_DIR = Path("../results/classification_validation_package")
VALIDATION_DIR.mkdir(exist_ok=True)

files_to_collect = [
    RESULTS_DIR / "model_comparison_v1.tsv",
    RESULTS_DIR / "cross_validation_results_v1.tsv",

    RESULTS_DIR / "test_predictions_logistic_regression_mlc_score_only.tsv",
    RESULTS_DIR / "test_predictions_logistic_regression_core_v2.tsv",
    RESULTS_DIR / "test_predictions_logistic_regression_extended_v2.tsv",
    RESULTS_DIR / "test_predictions_random_forest_core_v2.tsv",
    RESULTS_DIR / "test_predictions_random_forest_extended_v2.tsv",

    RESULTS_DIR / "feature_importance_random_forest_extended_v2.tsv",
    RESULTS_DIR / "coefficients_logistic_regression_extended_v2.tsv",

    PREPARED_DIR / "classification_core_features_report_v2.tsv",
    PREPARED_DIR / "classification_extended_features_report_v2.tsv",
    PREPARED_DIR / "classification_column_sets_v2.txt",
]

for src in files_to_collect:
    if src.exists():
        dst = VALIDATION_DIR / src.name
        dst.write_bytes(src.read_bytes())
        print("Copied:", src, "->", dst)
    else:
        print("Missing:", src)

print("\nValidation package created:")
print(VALIDATION_DIR)

Copied: ../results/classification_results/model_comparison_v1.tsv -> ../results/classification_validation_package/model_comparison_v1.tsv
Copied: ../results/classification_results/cross_validation_results_v1.tsv -> ../results/classification_validation_package/cross_validation_results_v1.tsv
Copied: ../results/classification_results/test_predictions_logistic_regression_mlc_score_only.tsv -> ../results/classification_validation_package/test_predictions_logistic_regression_mlc_score_only.tsv
Copied: ../results/classification_results/test_predictions_logistic_regression_core_v2.tsv -> ../results/classification_validation_package/test_predictions_logistic_regression_core_v2.tsv
Copied: ../results/classification_results/test_predictions_logistic_regression_extended_v2.tsv -> ../results/classification_validation_package/test_predictions_logistic_regression_extended_v2.tsv
Copied: ../results/classification_results/test_predictions_random_forest_core_v2.tsv -> ../results/classification_validati

In [53]:
import shutil

shutil.make_archive("../results/classification_validation_package", "zip", "../results/classification_validation_package")

print("Created: classification_validation_package.zip")

Created: classification_validation_package.zip
